# `02_yolo_seg_pipeline.ipynb` — Convert SAM1 Masks to YOLO26-seg Dataset

A two-step data conversion pipeline wrapping scripts from `src/yolo_seg/dataset/`.

## Directory Layout

```
data/
├── raw/ECUSTFD/
│   ├── JPEGImages/<stem>.JPG                 # Input images
│   └── ImageSets/Main/{trainval,test}.txt    # Paper-faithful split
└── processed/
    ├── sam_masks_full/masks/<stem>.npy       # Input masks from Step 01
    └── yolo_ecustfd_seg/                     # Output dataset
        ├── images/{train,val}/<stem>.JPG
        ├── labels/{train,val}/<stem>.txt     # YOLO polygon format
        └── ecustfd-seg.yaml                  # Dataset configuration
```

## Dataset Conventions

- **Paper-faithful split** (Liang & Li, 2017):
  - `train/`: 1,245 stems from `trainval.txt` (× 2 views ≈ 2,490 images)
  - `val/`: 1,733 stems from `test.txt` (× 2 views ≈ 3,466 pairs / 1,733 images)
  - Total 2,978 images without train/val overlap.
- **YOLO-seg label format**: `<class_idx> <x1> <y1> <x2> <y2> ...` normalized to $[0, 1]$.
- **Classes**: 20 total (19 food classes + 1 coin), alphabetically ordered (indices 0–19).
- **Contour extraction**: `cv2.findContours(RETR_EXTERNAL)` -> largest contour -> `cv2.approxPolyDP(epsilon=1.5)` -> normalized coordinates. Polygons with < 3 points or area < 9 pixels are discarded.

## Critical Checks
- Verify `sam_masks_full/masks/` contains all 2,978 masks before running conversion.


# Step 1 — Convert SAM1 `.npy` Masks to YOLO Polygon Labels

Executes `src/yolo_seg/dataset/convert_to_yolo_seg.py`:
- Parses `data/processed/sam_masks_full/masks/<stem>.npy`.
- Converts binary masks into normalized YOLO polygon coordinates.
- Splits into train/val using `trainval.txt` and `test.txt`.
- Copies images into `images/{train,val}/` and writes `labels/{train,val}/`.
- Generates `ecustfd-seg.yaml`.


In [1]:
# Cell 1 — Run convert_to_yolo_seg.py (paper-faithful split, no flags)
#
# Helper `run_script_step(...)`:
#   - chạy subprocess, stream từng dòng stdout ra console của cell này
#     (lưu vào lịch sử notebook) ĐỒNG THỜI ghi ra file log riêng
#     `data/processed/<step>/logs/run_<YYYYMMDD_HHMMSS>.log` để xem lại
#   - mỗi lần chạy tạo file log mới, KHÔNG overwrite log cũ
#   - tham số `step_tag` chỉ dùng để đặt tên thư mục logs/ cho dễ tra cứu
import subprocess, sys
from pathlib import Path
from datetime import datetime

PROJ = Path("E:/AI_Research/dlt8")

def run_script_step(rel_script: str, args: list[str], step_tag: str) -> int:
    """Run a python script with live streaming + per-run log file.

    Returns the process exit code.
    """
    script = PROJ / rel_script
    log_dir = PROJ / "data" / "processed" / step_tag / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"run_{ts}.log"

    header = (
        f"[run] {script.name} {' '.join(args)}\n"
        f"[cwd] {PROJ}\n"
        f"[log] {log_file}\n"
        f"[ts ] {datetime.now().isoformat(timespec='seconds')}\n"
        f"{'-' * 60}\n"
    )
    print(header, end="")
    with log_file.open("w", encoding="utf-8") as lf:
        lf.write(header)
        # Stream line-by-line so Jupyter shows progress AND file gets a copy
        proc = subprocess.Popen(
            [sys.executable, str(script), *args],
            cwd=str(PROJ),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            encoding="utf-8",
            errors="replace",
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            lf.write(line)
        rc = proc.wait()
        footer = f"{'-' * 60}\n[exit] {rc}\n"
        print(footer, end="")
        lf.write(footer)
    return rc

rc = run_script_step(
    "src/yolo_seg/dataset/convert_to_yolo_seg.py",
    [],  # defaults: --use-imagesets-split, no --force, no --dry-run
    step_tag="yolo_ecustfd_seg",
)
print(f"[exit] {rc}")

[run] convert_to_yolo_seg.py 
[cwd] E:\AI_Research\dlt8
[log] E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\logs\run_20260902_001154.log
[ts ] 2026-09-02T00:11:54
------------------------------------------------------------
ECUSTFD splits: trainval=1245 images, test=1733 images
Union: 2978 unique images
Train (trainval): 1245  |  Val (test): 1733

--- TRAIN ---
  Images written   : 1245
  Labels written   : 1245
  Skipped (no img) : 0
  Skipped (no poly): 0
  Total objects    : 2535
    apple: 156
    banana: 110
    bread: 30
    bun: 32
    coin: 1243
    doughnut: 88
    egg: 44
    fired_dough_twist: 58
    grape: 24
    lemon: 36
    litchi: 30
    mango: 123
    mooncake: 66
    orange: 118
    peach: 48
    pear: 72
    plum: 82
    qiwi: 71
    sachima: 54
    tomato: 50

--- VAL ---
  Images written   : 1733
  Labels written   : 1733
  Skipped (no img) : 0
  Skipped (no poly): 0
  Total objects    : 3527
    apple: 166
    banana: 102
    bread: 36
    bun: 58
    coin: 

# Step 2 — Verify YOLO Dataset Integrity

Executes `src/yolo_seg/dataset/verify_yolo_dataset.py` to validate:
1. `ecustfd-seg.yaml` exists and contains all 20 classes.
2. 1-to-1 matching between images and label files across splits.
3. Label coordinate bounds ($[0, 1]$) and polygon validity.
4. Per-class instance distributions.
5. Ultralytics `check_det_dataset` verification.


In [2]:
# Cell 2 — Run verify_yolo_dataset.py (sanity check)
#
# Helper `run_script_step(...)`:
#   - chạy subprocess, stream từng dòng stdout ra console của cell này
#     (lưu vào lịch sử notebook) ĐỒNG THỜI ghi ra file log riêng
#     `data/processed/<step>/logs/run_<YYYYMMDD_HHMMSS>.log` để xem lại
#   - mỗi lần chạy tạo file log mới, KHÔNG overwrite log cũ
#   - tham số `step_tag` chỉ dùng để đặt tên thư mục logs/ cho dễ tra cứu
import subprocess, sys
from pathlib import Path
from datetime import datetime

PROJ = Path("E:/AI_Research/dlt8")

def run_script_step(rel_script: str, args: list[str], step_tag: str) -> int:
    """Run a python script with live streaming + per-run log file.

    Returns the process exit code.
    """
    script = PROJ / rel_script
    log_dir = PROJ / "data" / "processed" / step_tag / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"run_{ts}.log"

    header = (
        f"[run] {script.name} {' '.join(args)}\n"
        f"[cwd] {PROJ}\n"
        f"[log] {log_file}\n"
        f"[ts ] {datetime.now().isoformat(timespec='seconds')}\n"
        f"{'-' * 60}\n"
    )
    print(header, end="")
    with log_file.open("w", encoding="utf-8") as lf:
        lf.write(header)
        # Stream line-by-line so Jupyter shows progress AND file gets a copy
        proc = subprocess.Popen(
            [sys.executable, str(script), *args],
            cwd=str(PROJ),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            encoding="utf-8",
            errors="replace",
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            lf.write(line)
        rc = proc.wait()
        footer = f"{'-' * 60}\n[exit] {rc}\n"
        print(footer, end="")
        lf.write(footer)
    return rc

rc = run_script_step(
    "src/yolo_seg/dataset/verify_yolo_dataset.py",
    [],
    step_tag="yolo_ecustfd_seg",
)
print(f"[exit] {rc}")


[run] verify_yolo_dataset.py 
[cwd] E:\AI_Research\dlt8
[log] E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\logs\run_20260902_001204.log
[ts ] 2026-09-02T00:12:04
------------------------------------------------------------
[OK] YAML exists: E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\ecustfd-seg.yaml
[OK] YAML classes: 20 (expected 21)

[train] images=1245 labels=1245
[OK] [train] all images have matching labels
[OK] [train] first 20 label files: all valid
[OK] Image sample: apple001S(1).JPG shape=(551, 816, 3)
[OK] Image sample: apple001S(2).JPG shape=(587, 816, 3)
[OK] Image sample: apple001T(1).JPG shape=(612, 816, 3)
[train] total objects: 2535
    class  0 (apple               ):  156 objects
    class  1 (banana              ):  110 objects
    class  2 (bread               ):   30 objects
    class  3 (bun                 ):   32 objects
    class  4 (coin                ): 1243 objects
    class  5 (doughnut            ):   88 objects
    class  6 (egg           

# Pipeline Completion

Dataset generated at `data/processed/yolo_ecustfd_seg/`:
- `images/{train,val}/`
- `labels/{train,val}/`
- `ecustfd-seg.yaml`

Proceed to training in `03_yolo_seg_train.ipynb`.
